In [1]:
import cv2
from ultralytics import YOLO

# Load the model
yolo = YOLO('yolo_model_trined_on_head.pt')


In [3]:
# Function to get class colors
def getColours(cls_num):
    base_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
    color_index = cls_num % len(base_colors)
    increments = [(1, -2, 1), (-2, 1, -1), (1, -1, 2)]
    color = [base_colors[color_index][i] + increments[color_index][i] * 
    (cls_num // len(base_colors)) % 256 for i in range(3)]
    return tuple(color)

In [ ]:
import cv2
import time
video_path = "E:/manorath Leriya/ras/1/VTS_02_2.VOB"  # Replace with your actual video path
videoCap = cv2.VideoCapture("https://cmn.cloudvms.in/cam2.mp4")

while True:
    ret, frame = videoCap.read()
    if not ret:
        break  # Exit loop when the video ends
    t = time.time()
    try:
        results = yolo.track(frame, stream=True ,device='cuda', persist=True)
        tx = time.time() - t
        # print(f"YOLO tracking time: {tx:.2f} seconds")
        fps = 1 / tx
        print(f"FPS: {fps:.2f}")
    except Exception as e:
        print(f"Error in YOLO tracking: {e}")
        continue

    for result in results:
        classes_names = result.names
        track_ids = result.boxes.id.int().cpu().tolist()

        for box in result.boxes:
            if box.conf[0] > 0.4:
                [x1, y1, x2, y2] = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cls = int(box.cls[0])
                class_name = classes_names[cls]
                colour = getColours(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f'{class_name} {box.conf[0]:.2f}', (x1, y1),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, colour, 2)

    cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

videoCap.release()
cv2.destroyAllWindows()


Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
FPS: 1003.66

0: 384x640 (no detections), 50.7ms
Speed: 3.1ms preprocess, 50.7ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
FPS: 1005.11

0: 384x640 (no detections), 55.8ms
Speed: 3.7ms preprocess, 55.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zero
FPS: 1002.94

0: 384x640 (no detections), 47.9ms
Speed: 3.2ms preprocess, 47.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
Error in YOLO tracking: float division by zero
Error in YOLO tracking: float division by zer

In [11]:
from collections import defaultdict
import numpy as np
cap = cv2.VideoCapture("D:/Chrome download/cam2new.mp4")

# Store the track history
track_history = defaultdict(lambda: [])

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO11 tracking on the frame, persisting tracks between frames
        result = yolo.track(frame,device="cuda", persist=True)[0]

        # Get the boxes and track IDs
        if result.boxes and result.boxes.is_track:
            boxes = result.boxes.xywh.cpu()
            track_ids = result.boxes.id.int().cpu().tolist()

            # Visualize the result on the frame
            frame = result.plot()

            # Plot the tracks
            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))  # x, y center point
                if len(track) > 30:  # retain 30 tracks for 30 frames
                    track.pop(0)

                # Draw the tracking lines
                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

        # Display the annotated frame
        cv2.imshow("YOLO11 Tracking", frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

0: 384x640 1 Human head, 67.0ms
Speed: 5.7ms preprocess, 67.0ms inference, 7.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 40.9ms
Speed: 5.5ms preprocess, 40.9ms inference, 7.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 39.7ms
Speed: 4.6ms preprocess, 39.7ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 19.3ms
Speed: 3.9ms preprocess, 19.3ms inference, 5.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 24.2ms
Speed: 4.7ms preprocess, 24.2ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 19.1ms
Speed: 2.7ms preprocess, 19.1ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 18.6ms
Speed: 2.8ms preprocess, 18.6ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Human head, 18.6ms
Speed: 2.9ms preprocess, 18.6ms inference, 5.3ms postproce

In [7]:
track_history

defaultdict(<function __main__.<lambda>()>,
            {601: [(824.0684204101562, 169.31610107421875),
              (820.1160888671875, 166.62408447265625),
              (808.425537109375, 158.49526977539062),
              (799.056396484375, 153.51800537109375),
              (789.9761352539062, 150.85604858398438),
              (780.6710815429688, 149.03591918945312),
              (767.180908203125, 147.08387756347656),
              (755.21044921875, 146.14903259277344),
              (744.11962890625, 145.72337341308594),
              (732.8387451171875, 144.68667602539062)],
             599: [(713.1433715820312, 137.88381958007812),
              (707.3345336914062, 135.27853393554688),
              (700.9078369140625, 132.68710327148438),
              (695.2178955078125, 127.46482849121094)],
             653: [(654.4566650390625, 119.41283416748047),
              (651.47412109375, 117.67170715332031),
              (649.0028076171875, 115.06585693359375),
             